# uy_tools - utilidades (notebook)
Funciones que consumen APIs públicas: Open-Meteo, CoinGecko y Public-APIs README.

# uy_tools - utilidades para coches de segunda mano
Funciones útiles para el flujo de comprobar y probar coches: clima histórico por fecha, conversiones de moneda, evaluación de si es buen día para una prueba, y un hook hacia el RAG del proyecto.

In [ ]:
import httpx
from typing import Any, Dict, List, Optional
from datetime import datetime, date, timedelta
import math

## Clima actual y clima histórico por fecha
Usamos la API de Open-Meteo (forecast para ahora, archive/era5 para histórico).

In [ ]:
def get_weather(latitude: float, longitude: float) -> Dict[str, Any]:
    try:
        url = "https://api.open-meteo.com/v1/forecast"
        params = {"latitude": latitude, "longitude": longitude, "current_weather": True}
        r = httpx.get(url, params=params, timeout=10.0)
        r.raise_for_status()
        data = r.json()
        return {"success": True, "data": data.get("current_weather"), "raw": data}
    except Exception as e:
        return {"success": False, "error": str(e)}

In [ ]:
def get_weather_on_date(latitude: float, longitude: float, day: str) -> Dict[str, Any]:
    """Return basic weather summary for a given date (YYYY-MM-DD) using Open-Meteo archive (ERA5)."""
    try:
        # ensure date format
        d = datetime.fromisoformat(day).date() if isinstance(day, str) else day
        start = d.isoformat()
        end = d.isoformat()
        url = "https://archive-api.open-meteo.com/v1/era5"
        params = {"latitude": latitude, "longitude": longitude, "start_date": start, "end_date": end, "hourly": "temperature_2m,precipitation,winddirection_10m,windspeed_10m"}
        r = httpx.get(url, params=params, timeout=20.0)
        r.raise_for_status()
        data = r.json()
        # compute simple summary: max/min/total precipitation and average windspeed
        hourly = data.get('hourly', {})
        temps = hourly.get('temperature_2m', [])
        prec = hourly.get('precipitation', [])
        wind = hourly.get('windspeed_10m', [])
        summary = {}
        if temps:
            summary['temp_min'] = min(temps)
            summary['temp_max'] = max(temps)
            summary['temp_avg'] = sum(temps)/len(temps)
        if prec:
            summary['precip_total_mm'] = sum(prec)
        if wind:
            summary['wind_avg_kmh'] = (sum(wind)/len(wind))
        return {"success": True, "date": start, "summary": summary, "raw": data}
    except Exception as e:
        return {"success": False, "error": str(e)}

## Conversión de moneda
Funciones para convertir entre monedas y convertir a múltiples objetivos. Usamos exchangerate.host (sin API key).

In [ ]:
def convert_currency(amount: float, from_currency: str, to_currency: str, date: Optional[str] = None) -> Dict[str, Any]:
    try:
        params = {"from": from_currency.upper(), "to": to_currency.upper(), "amount": amount}
        if date:
            params['date'] = date
        url = "https://api.exchangerate.host/convert"
        r = httpx.get(url, params=params, timeout=10.0)
        r.raise_for_status()
        data = r.json()
        return {"success": True, "query": data.get('query'), "result": data.get('result'), "info": data.get('info')}
    except Exception as e:
        return {"success": False, "error": str(e)}

In [ ]:
# Coin price helper using CoinGecko public API
def get_coin_price(coin_id: str, vs_currency: str = 'usd') -> Dict[str, Any]:
    try:
        url = f'https://api.coingecko.com/api/v3/simple/price'
        params = {'ids': coin_id, 'vs_currencies': vs_currency, 'include_market_cap': 'true'}
        r = httpx.get(url, params=params, timeout=8.0)
        r.raise_for_status()
        data = r.json()
        return {'success': True, 'coin': coin_id, 'data': data.get(coin_id, {})}
    except Exception as e:
        return {'success': False, 'error': str(e)}

# Simple public-apis README search (fetch raw and do a substring match)
def search_public_apis(query: str, max_results: int = 10) -> Dict[str, Any]:
    try:
        url = 'https://raw.githubusercontent.com/public-apis/public-apis/master/README.md'
        r = httpx.get(url, timeout=10.0)
        r.raise_for_status()
        text = r.text.lower()
        q = query.lower()
        lines = [l for l in text.splitlines() if q in l][:max_results]
        return {'success': True, 'query': query, 'results': lines}
    except Exception as e:
        return {'success': False, 'error': str(e)}

## Evaluación de si es buen día para probar un coche
Función `assess_testability` que combina la información meteorológica para dar una recomendación.

In [ ]:
def assess_testability(day: str, latitude: float, longitude: float) -> Dict[str, Any]:
    """Return an assessment whether it's a good day to test-drive a used car at the given date and location."""
    w = get_weather_on_date(latitude, longitude, day)
    if not w.get('success'):
        return { 'success': False, 'error': 'Could not retrieve weather: ' + str(w.get('error')) }
    summary = w.get('summary', {})
    reasons = []
    score = 0
    # precipitation rule
    prec = summary.get('precip_total_mm', 0)
    if prec and prec > 0.5:
        reasons.append(f'Precipitation total {prec} mm — not ideal for testing')
        score -= 2
    # wind rule
    wind = summary.get('wind_avg_kmh', 0)
    if wind and wind > 40:
        reasons.append(f'High wind average {wind:.1f} km/h — caution')
        score -= 1
    # temperature extremes
    tmin = summary.get('temp_min')
    tmax = summary.get('temp_max')
    if tmin is not None and tmin < -5:
        reasons.append(f'Low temperature {tmin}°C — may affect battery/startup')
        score -= 1
    if tmax is not None and tmax > 40:
        reasons.append(f'High temperature {tmax}°C — caution with engines/AC')
        score -= 1
    # simple heuristic: if score >=0 OK, else Not ideal
    recommendation = 'OK' if score >= 0 and not reasons else 'Not ideal' if score < 0 else 'Caution'
    return { 'success': True, 'date': day, 'recommendation': recommendation, 'reasons': reasons, 'summary': summary }

## Hook RAG
Función `rag_query` que integra con el servicio RAG si está disponible - intenta una llamada HTTP local o devuelve un mensaje útil.

In [ ]:
# RAG query using central rag_config.py. Change rag_config.RAG_HTTP_ENDPOINT or RAG_PYTHON_CLIENT to enable real RAG.
try:
    from rag_config import RAG_HTTP_ENDPOINT, RAG_PYTHON_CLIENT, RAG_HTTP_TIMEOUT
except Exception:
    RAG_HTTP_ENDPOINT = 'http://localhost:8000/rag'
    RAG_PYTHON_CLIENT = 'rag_client'
    RAG_HTTP_TIMEOUT = 6.0

def rag_query(query: str) -> Dict[str, Any]:
    """Query the configured RAG integration."""
    # 1) Try HTTP endpoint if configured
    if RAG_HTTP_ENDPOINT:
        try:
            r = httpx.post(RAG_HTTP_ENDPOINT, json={ 'query': query }, timeout=RAG_HTTP_TIMEOUT)
            r.raise_for_status()
            return { 'success': True, 'source': 'http', 'endpoint': RAG_HTTP_ENDPOINT, 'response': r.json() }
        except Exception as e:
            http_err = str(e)
    else:
        http_err = 'RAG_HTTP_ENDPOINT not set'

    # 2) Try Python client if available
    if RAG_PYTHON_CLIENT:
        try:
            rc = __import__(RAG_PYTHON_CLIENT)
            if hasattr(rc, 'query'):
                resp = rc.query(query)
                return { 'success': True, 'source': 'python_client', 'client': RAG_PYTHON_CLIENT, 'response': resp }
            else:
                py_err = f'python client {RAG_PYTHON_CLIENT} has no query()'
        except Exception as e:
            py_err = str(e)
    else:
        py_err = 'RAG_PYTHON_CLIENT not set'

    # Fallback: informative message so the UI can show what to configure
    return { 'success': False, 'error': 'RAG backend not available locally', 'http_error': http_err, 'python_error': py_err, 'hint': 'Set rag_config.RAG_HTTP_ENDPOINT or RAG_PYTHON_CLIENT to enable RAG' }

## Ejemplos / ejercicios ampliados
Ejemplos orientados a coches de segunda mano: clima histórico para la fecha de prueba, conversión de precio y evaluación de si es un buen día para probar.

In [ ]:
# Nota: Ejemplos interactivos removidos para simplificar el notebook; las funciones están disponibles para importar desde uy_tools.ipynb ejecución de celdas de código.